# 03 Feature Selection

Notebook 03 reduces the preprocessed feature set before final modeling. It removes weak or redundant features and keeps columns that are more likely to generalize to unseen test rows.

The final production script uses a cleaner hand-selected raw-column configuration, but this notebook documents the reasoning that led there.


## Purpose Of This Notebook

Feature selection is used to make the model smaller, cleaner, and easier to explain. The notebook checks for low-value columns, redundant columns, and columns that do not contribute meaningfully in a baseline model.

This step is exploratory: it informs the final selected raw columns used by the production script.


In [ ]:
# 1) Setup.
# Reuse the preprocessing output from the previous notebook.
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "notebooks" / "02_preprocessing.ipynb").exists():
    project_root = project_root.parent

preprocessing_notebook = project_root / "notebooks" / "02_preprocessing.ipynb"

# Running the preprocessing notebook creates X_train, y_train, and X_test in the
# current notebook namespace.
%run "$preprocessing_notebook"

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

required_objects = [
    "X_train",
    "y_train",
    "X_test"
]

# Fail early if preprocessing did not produce the expected objects.
for obj in required_objects:
    if obj not in globals():
        raise NameError(
            f"{obj} does not exist. Run Notebook 2 before running this notebook."
        )

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)


## Feature Quality Overview

Before training models, this section checks each feature for missing values, unique values, and data type. These simple diagnostics often reveal columns that are constant, mostly empty, or too noisy.


In [ ]:
# 2) Basic feature quality check.
# Combine several quality signals so weak features are easy to spot.
feature_quality = pd.DataFrame({
    "missing_values": X_train.isna().sum(),
    "unique_values": X_train.nunique(),
    "dtype": X_train.dtypes.astype(str)
})

# Sorting by unique values puts constants and very low-variation features first.
feature_quality = feature_quality.sort_values(
    "unique_values",
    ascending=True
)

print("features:", X_train.shape[1])
print("missing values in X_train:", X_train.isna().sum().sum())
print("non-numeric columns:", len(X_train.select_dtypes(exclude="number").columns))

display(feature_quality.head(20))


In [ ]:
# 3) Constant and near-constant features.
# Near-constant features usually add little information and can make the model noisier.
near_constant_threshold = 0.99

# Constant columns carry no variation, so they cannot help distinguish target classes.
constant_features = X_train.columns[
    X_train.nunique() <= 1
].tolist()

near_constant_features = []

for col in X_train.columns:
    # The most common value ratio measures how dominated a feature is by one value.
    most_common_ratio = X_train[col].value_counts(normalize=True).iloc[0]

    if most_common_ratio >= near_constant_threshold:
        near_constant_features.append(col)

near_constant_features = sorted(set(near_constant_features))

print("constant features:", len(constant_features))
print("near-constant features:", len(near_constant_features))

display(
    pd.DataFrame({
        "feature": near_constant_features
    }).head(30)
)


## Redundancy Checks

Highly correlated features can repeat the same signal. Removing one feature from a strongly correlated pair keeps the model simpler and easier to interpret.


In [ ]:
# 4) Correlation check.
# Highly correlated features often carry duplicate information.
correlation_threshold = 0.95
# A high threshold removes only very redundant numeric relationships.

corr_matrix = X_train.corr(numeric_only=True).abs()

# Keep only the upper triangle so each feature pair appears once.
upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
    .reset_index()
)

high_corr_pairs.columns = [
    "feature_1",
    "feature_2",
    "correlation"
]

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["correlation"] >= correlation_threshold
]

print("highly correlated feature pairs:", len(high_corr_pairs))

display(high_corr_pairs.head(30))


In [ ]:
# 5) Choose correlated features to remove.
# For each highly correlated pair, remove the feature with fewer unique values as
# a simple rule for keeping the more detailed signal.
features_to_remove_from_correlation = set()

for _, row in high_corr_pairs.iterrows():
    feature_1 = row["feature_1"]
    feature_2 = row["feature_2"]

    # Skip pairs where one feature has already been selected for removal.
    if feature_1 in features_to_remove_from_correlation:
        continue

    if feature_2 in features_to_remove_from_correlation:
        continue

    unique_1 = X_train[feature_1].nunique()
    unique_2 = X_train[feature_2].nunique()

    if unique_1 <= unique_2:
        features_to_remove_from_correlation.add(feature_1)
    else:
        features_to_remove_from_correlation.add(feature_2)

features_to_remove_from_correlation = sorted(
    features_to_remove_from_correlation
)

print("features selected for removal due to correlation:", len(features_to_remove_from_correlation))

display(
    pd.DataFrame({
        "feature": features_to_remove_from_correlation
    }).head(30)
)


## Feature Importance Check

A baseline tree model is used here only as a diagnostic tool. Features with very low or zero importance are candidates for removal, but the final decision also considers leakage and interpretability.


In [ ]:
# 6) Train/validation split for feature importance.
# The random forest estimates feature usefulness on training rows and is checked
# on a held-out validation split.
X_train_part, X_valid_part, y_train_part, y_valid_part = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("X_train_part shape:", X_train_part.shape)
print("X_valid_part shape:", X_valid_part.shape)
print("y_train_part shape:", y_train_part.shape)
print("y_valid_part shape:", y_valid_part.shape)


In [ ]:
# 7) Feature importance from baseline model.
# Random forest importance is used as a practical screening tool, not as the final model.
importance_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

# Fit only on the training portion so the validation score remains honest.
importance_model.fit(
    X_train_part,
    y_train_part
)

valid_predictions = importance_model.predict(X_valid_part)

valid_accuracy = accuracy_score(
    y_valid_part,
    valid_predictions
)

print("validation accuracy of feature-importance model:", round(valid_accuracy, 4))


In [ ]:
# 8) Display feature importance.
# Higher values mean the random forest used that feature more often to reduce impurity.
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importance_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

display(feature_importance.head(30))


In [ ]:
# 9) Select final features.
# This selected feature list connects preprocessing to model comparison.

# Features with zero measured importance are candidates for removal.
zero_importance_features = (
    feature_importance[
        feature_importance["importance"] == 0
    ]["feature"]
    .tolist()
)

# Combine all removal rules into one list so each feature is removed only once.
features_to_remove = sorted(
    set(near_constant_features)
    | set(features_to_remove_from_correlation)
    | set(zero_importance_features)
)

selected_features = [
    col for col in X_train.columns
    if col not in features_to_remove
]

print("original features:", X_train.shape[1])
print("features removed:", len(features_to_remove))
print("selected features:", len(selected_features))

removed_features_summary = pd.DataFrame({
    "feature": features_to_remove
})

display(removed_features_summary.head(40))


## Selected Dataset Creation

After deciding which features to keep, the notebook creates reduced training and test matrices. The assertions confirm that the two matrices still line up correctly.


In [ ]:
# 10) Create selected datasets.
# Reduced matrices for the model-comparison notebook.
X_train_selected = X_train[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

# Assertions protect against accidentally dropping different columns in train and test.
assert X_train_selected.shape[1] == X_test_selected.shape[1]
assert X_train_selected.isna().sum().sum() == 0
assert X_test_selected.isna().sum().sum() == 0
assert len(X_train_selected.select_dtypes(exclude="number").columns) == 0
assert len(X_test_selected.select_dtypes(exclude="number").columns) == 0

print("X_train_selected shape:", X_train_selected.shape)
print("y_train shape:", y_train.shape)
print("X_test_selected shape:", X_test_selected.shape)


## Feature Selection Summary

This final table documents how many features were removed at each stage. It helps explain that the final model was not trained on every raw column blindly.


In [ ]:
# 11) Final feature-selection summary.
# Summarize the reduction process for easy reporting.
selection_summary = pd.DataFrame({
    "step": [
        "original features",
        "near-constant features removed",
        "correlated features removed",
        "zero-importance features removed",
        "final selected features"
    ],
    "count": [
        X_train.shape[1],
        len(near_constant_features),
        len(features_to_remove_from_correlation),
        len(zero_importance_features),
        len(selected_features)
    ]
})

display(selection_summary)

# Show the strongest remaining features after the removal rules have been applied.
top_selected_features = (
    feature_importance[
        feature_importance["feature"].isin(selected_features)
    ]
    .head(25)
    .reset_index(drop=True)
)

display(top_selected_features)
